# Spark Analytics Gallery - Hubs Cargo USGC Swaps vs Spark30S

Here we plot USGC Swaps data (Hubs Cargo API) vs Spark30S Atlantic freight rates.

For a full explanation of how to import our Hubs Cargo or Spark30S data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html

### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint:
https://www.sparkcommodities.com/api/

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

__This section can remain unchanged for most Spark API users.__

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.

The code then prints the available prices that are callable from the API, and their corresponding Python ticker names are displayed as a list at the bottom of the Output.

In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

# 2. USGC Swaps Data

Here we define the functions used to call the USGC Swaps data from the Hubs Cargo API. For more information on the Hubs Cargo API, please visit the API website or download our Hubs Cargo code sample:

https://www.sparkcommodities.com/api/hubs/lng-cargo.html

In [ ]:

## Defining the function

def fetch_ref_data(access_token, format='json'):

    uri="v1.0/lng/hubs/cargo/reference-data/"
    print(uri)
    
    content = do_api_get_query(
        uri=uri, access_token=access_token, format=format
    )
    
    if format == 'json':
        my_dict = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            my_dict = content
            print('No Data to Show')
        else:
            my_dict = content.decode('utf-8')
            my_dict = pd.read_csv(StringIO(my_dict))  # automatically converting into a Pandas DataFrame when choosing CSV format
    
    return my_dict

ref_df = fetch_ref_data(access_token, format='csv')
ref_df.head(5)

In [ ]:
# Fetch Live posts
def fetch_active_posts(access_token, incoterm=None, region=None, format='json'):

    query_params = ''

    if incoterm is not None:
        query_params += '?incoterm={}'.format(incoterm)
    if region is not None:
        if len(query_params) >0:
            query_params += '&region={}'.format(region)
        else:
            query_params += '?region={}'.format(region)


    uri="v1.0/lng/hubs/cargo/active/{}".format(query_params)
    print(uri)
    
    content = do_api_get_query(
        uri=uri, access_token=access_token, format=format
    )
    
    if format == 'json':
        my_dict = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            my_dict = content
            print('No Data to Show')
        else:
            my_dict = content.decode('utf-8')
            my_dict = pd.read_csv(StringIO(my_dict))  # automatically converting into a Pandas DataFrame when choosing CSV format
    
    return my_dict

# Fetch Historical Posts
def fetch_inactive_posts(access_token, incoterm=None, region=None, format='json'):

    query_params = ''

    if incoterm is not None:
        query_params += '?incoterm={}'.format(incoterm)
    if region is not None:
        if len(query_params) >0:
            query_params += '&region={}'.format(region)
        else:
            query_params += '?region={}'.format(region)


    uri="v1.0/lng/hubs/cargo/inactive/{}".format(query_params)
    print(uri)
    
    content = do_api_get_query(
        uri=uri, access_token=access_token, format=format
    )
    
    if format == 'json':
        my_dict = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            my_dict = content
            print('No Data to Show')
        else:
            my_dict = content.decode('utf-8')
            my_dict = pd.read_csv(StringIO(my_dict))  # automatically converting into a Pandas DataFrame when choosing CSV format
    
    return my_dict

In [ ]:
# calling the live data
usgc_livedf = fetch_active_posts(access_token, incoterm='fob', region='usg', format='csv')

# calling the historical data
usgc_histdf = fetch_inactive_posts(access_token, incoterm='fob', region='usg', format='csv')

# Combining the dataframes to create a complete dataset
total_df = pd.concat([usgc_livedf, usgc_histdf])

# Converting dates to datetimes
total_df['PostedAtUTC'] = pd.to_datetime(total_df['PostedAtUTC'])
total_df['WindowStartDate'] = pd.to_datetime(total_df['WindowStartDate'])
total_df['WindowEndDate'] = pd.to_datetime(total_df['WindowEndDate'])
total_df['PostedDate'] = total_df['PostedAtUTC'].dt.strftime('%Y-%m-%d')

# Creating a swaps-only dataframe
swaps = total_df[total_df['InterestType'] == 'swap'].copy()

swaps.head(5)

# 3. Calling Spark30S Freight data

Here we define the functions used to call the Spark30S Freight data from the Freight Contracts API. For more information on the Freight Contracts API, please visit the API website or download our Freight Contracts code sample:

https://www.sparkcommodities.com/api/lng-freight/contracts.html

In [ ]:
# Defining function to fetch historical freight data from API
def fetch_historical_price_releases(access_token, ticker, limit, offset=None, vessel=None):

    query_params = "?limit={}".format(limit)
    if offset is not None:
        query_params += "&offset={}".format(offset)
    
    if vessel is not None:
        query_params += "&vessel-type={}".format(vessel)

    content = do_api_get_query(
        uri="/v1.0/contracts/{}/price-releases/{}".format(ticker, query_params),
        access_token=access_token,
    )

    uri="/v1.0/contracts/{}/price-releases/{}".format(ticker, query_params)
    print(uri)

    my_dict = content["data"]

    return my_dict


# Defining the function for storing and formatting the data into a Pandas DataFrame
def store_and_format(dict_hist):
    stored_data = {
        "ReleaseDate": [],
        "Ticker": [],
        "PeriodStart": [],
        "USDperday": [],
        "USDperdayMax": [],
        "USDperdayMin": [],
    }

    for release in dict_hist:

        data_points = release["data"][0]["dataPoints"]

        for data_point in data_points:
            stored_data['Ticker'].append(release["contractId"])
            stored_data['ReleaseDate'].append(release["releaseDate"])

            period_start_at = data_point["deliveryPeriod"]["startAt"]
            stored_data['PeriodStart'].append(period_start_at)

            stored_data['USDperday'].append(data_point["derivedPrices"]["usdPerDay"]["spark"])
            stored_data['USDperdayMin'].append(data_point["derivedPrices"]["usdPerDay"]["sparkMin"])
            stored_data['USDperdayMax'].append(data_point["derivedPrices"]["usdPerDay"]["sparkMax"])
            
    historical_df = pd.DataFrame(stored_data)
        
    historical_df["USDperday"] = pd.to_numeric(historical_df["USDperday"])
    historical_df["USDperdayMax"] = pd.to_numeric(historical_df["USDperdayMax"])
    historical_df["USDperdayMin"] = pd.to_numeric(historical_df["USDperdayMin"])

    historical_df["ReleaseDate"] = pd.to_datetime(historical_df["ReleaseDate"])
        
    return historical_df

In [ ]:
# determining approximate length of freight data needed based on swaps date range
freight_length = (swaps['PostedAtUTC'].max() - swaps['PostedAtUTC'].min()).days

In [ ]:
### Calling Spark30S data using above functions
spark30s_raw = fetch_historical_price_releases(access_token, 'spark30s', limit=freight_length, offset=0)
spark30s = store_and_format(spark30s_raw)

spark30s.head(5)

# 4. Computing Swaps "Momentum" metric

Here we define a function to compute the "momentum" metric for the swaps data. This iterates through all swap posts to classify them as either "Request for Later Loading Slot" or "Request for Earlier Loading Slot".

The function includes several options to filter the data & which posts to include within this metric

In [ ]:

def compute_swap_momentum(posts_df, spot_window=False, window_days=30,
                          normalise=False, min_diff=0, max_diff=None):
    f"""
    Parameters
    ----------
    spot_window : bool, default False
        True -> a post counts only if EITHER its offered or requested date sits within the (T -> T+"window_days") window, 
                where  T = the post's PostedAt date).
    window_days : int, default 30
        The length of window to apply for the "spot_window" param (T+30 by default to match Spark30S spec; T = the post's PostedAt date).
    normalise : bool, default False
        Normalise for the amount of posts per month
    min_diff : int, default 0
        Only count posts whose |requested - offered| shift is >= min_diff days. 0 = no filtering.
    max_diff : int or None, default None
        Only count posts whose |requested - offered| shift is <= min_diff days. 0 = no filtering.

    Returns
    -------
    moments_df = momentum metric per release date
    months_df = momentum metric summed over the month
    """

    # Creating a copy of the dataframe, and making sure it only has Swaps
    df = posts_df.copy()
    df = df[df['InterestType'] == 'swap']

    df['PostedAtUTC'] = pd.to_datetime(df['PostedAtUTC'])
    df['WindowStartDate'] = pd.to_datetime(df['WindowStartDate'])
    df['PostedDate'] = df['PostedAtUTC'].dt.strftime('%Y-%m-%d')

    # Initiating lists
    datetimes, later, earlier = [], [], []

    # For each post date
    for r in df['PostedDate'].unique():
        rdf = df[df['PostedDate'] == r]        # filter dataframe per post date
        l = e = 0                               # initiate later and earlier counters

        t_start = pd.to_datetime(r)
        t_end = t_start + pd.Timedelta(days=window_days)

        for pid in rdf['PostID'].unique():              # Iterating through every post
            tdf = rdf[rdf['PostID'] == pid]
            offered = tdf[tdf['OrderType'] == 'offered']
            requested = tdf[tdf['OrderType'] == 'requested']
            if offered.empty or requested.empty:        # Check if empty
                continue

            offer = offered['WindowStartDate'].min()    # earliest offered
            sell = requested['WindowStartDate'].min()    # earliest requested

            if spot_window:                             # Apply spot_window filtering if specified
                offer_in = t_start <= offer <= t_end       
                sell_in = t_start <= sell <= t_end
                if not (offer_in or sell_in):
                    continue
                                          
            shift = abs((sell - offer).days)                # Apply min_diff or max_diff if specified
            if shift < min_diff:
                continue
            if max_diff is not None and shift > max_diff:
                continue

            if offer < sell:                                # Counting each post as "earlier" or "later" loading
                l += 1
            elif sell < offer:
                e += 1

        datetimes.append(rdf['PostedAtUTC'].iloc[0])
        later.append(l)
        earlier.append(e)

    def _diff(later_s, earlier_s):                          # Creating an internal function to calculate metric
        net = later_s - earlier_s
        if normalise:                                       # Applying normalisation if specified
            total = later_s + earlier_s
            return np.where(total > 0, net / total, np.nan)
        return net

    moments_df = pd.DataFrame({                             # Creating dataframe with per-post_date metrics
        'Date': pd.to_datetime(datetimes),
        'Later': later,
        'Earlier': earlier,
    })
    moments_df['Diff'] = _diff(moments_df['Later'], moments_df['Earlier'])
    moments_df = moments_df.sort_values('Date', ascending=False).reset_index(drop=True)
    moments_df['Month'] = moments_df['Date'].dt.strftime('%b%y')
    moments_df = moments_df[['Date', 'Month', 'Earlier', 'Later', 'Diff']]

    months_df = (                                          # Summing by month
        moments_df
        .groupby('Month', as_index=False)[['Earlier', 'Later']]
        .sum()
    )
    months_df['Diff'] = _diff(months_df['Later'], months_df['Earlier'])
    months_df['MonthDatetime'] = pd.to_datetime(months_df['Month'], format='%b%y')
    months_df = months_df.sort_values('MonthDatetime').reset_index(drop=True)
    months_df = months_df[['Month', 'MonthDatetime', 'Earlier', 'Later', 'Diff']]

    return moments_df, months_df

In [ ]:
# Applying function to swaps dataframe 
moments_df, months_df = compute_swap_momentum(swaps)

# Preview per-post_date data 
moments_df.head(3)

In [ ]:
# Preview monthly data
months_df

# 5. Plotting

Here we plot the Spark30S Freight rates vs the Monthly USGC Swaps "Momentum". 

__N.B.__ We use the "months_df" dataframe defined in the cells above, where no additional filter parameters were specified when calling the "compute_swap_momentum" function.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

fig1, ax1 = plt.subplots(figsize=(16,7))

# creating two separate dataframe views to plot "Net Later" and "Net Earlier" posts in separate colours 
pos_df = months_df[months_df['Diff'] >0]
neg_df = months_df[months_df['Diff'] <0]

# Plotting thicker x-axis line
ax1.hlines(0, months_df['MonthDatetime'].iloc[1] - pd.Timedelta(days=60), months_df['MonthDatetime'].iloc[-1] + pd.Timedelta(days=60), color='grey')

# Plotting Swaps momentum metric bar charts
ax1.bar(pos_df['MonthDatetime'] + pd.Timedelta(days=15), pos_df['Diff'], color='mediumseagreen', width=27, align='center', label='Later')
ax1.bar(neg_df['MonthDatetime'] + pd.Timedelta(days=15), neg_df['Diff'], color='firebrick', width=27, align='center', label='Earlier')
plt.legend(loc='lower right')

# Limiting date range to match swaps data
ax1.set_xlim(months_df['MonthDatetime'].iloc[0] - pd.Timedelta(days=10), months_df['MonthDatetime'].iloc[-1] + pd.Timedelta(days=40))
ax1.set_ylim(-15, 10)

# Creating twin-axis to plot Spark30S freight rates
ax12 = ax1.twinx()

# Plotting Spark30S
ax12.plot(spark30s['ReleaseDate'], spark30s['USDperday'], linewidth=2, color='mediumblue', alpha=0.8, zorder=0, label='Spark30S (Atlantic)')
ax12.scatter(spark30s['ReleaseDate'].iloc[0], spark30s['USDperday'].iloc[0], color='mediumblue', marker='o', alpha=0.8, s=80)

ax12.grid(False)
plt.legend(loc='upper right')